# DataWhisperer: Talk to Your CSV
Single-notebook build. Run cells top to bottom, in order, in a **fresh runtime** (Runtime > Restart session first if you've run this before).

**Stack:** LangChain (raw PythonAstREPLTool + ReAct agent via langchain_classic), Groq (Llama 3.3 70B), Pandas, Streamlit, ngrok tunnel.

**Outcomes covered:**
1. Answers analytical questions by generating + running pandas code
2. Correct charts for 3+ question types
3. Safety sandbox (agent's code can only see df/pd/plt) + explains the code it ran
4. Conversation memory (follow-up questions work without repeating context)
5. Auto data profiling on upload (shape, dtypes, missing values, stats)

**fixes/things that broke while testing:**
- langchain 1.0 moved AgentExecutor and create_react_agent into langchain_classic.agents, took me a while to figure out why the old import was failing
- started with gemini but gemini-2.5-flash got deprecated for new api keys mid project, switched to groq (llama 3.3 70b) instead - free key, no billing wall
- max_iterations was 6 first, chart questions kept dying with "stopped due to iteration limit" so bumped it to 25, also added max_execution_time=120 and early_stopping_method="generate"
- for "excluding X" type questions the agent was using .count() on the whole thing instead of filtering first, added a rule to filter with boolean indexing before aggregating - tested with the strike rate question, now gives the right number (237.31)
- had to add plt.close('all') before every question because streamlit reruns the whole script but doesnt restart python, so old charts/figures were sticking around
- annoying bug: agent would save the chart as something like dismissal_pie_chart.png instead of chart.png like i told it to, so streamlit never found it to display. fixed by repeating the filename instruction harder + added a fallback that just grabs whatever png was created most recently
- bigger bug: tested with a different csv (nz alcohol data) and asked for "beer" totals, got back 0.0. turned out the exact category "Beer" only had data up to 1981, real recent numbers were under "Total beer" instead. same thing broke a chart request too - it filtered to an empty dataframe and just plotted nothing. added a rule so the agent checks for a similar/overlapping category name (and doesnt plot an empty df) before giving a final answer instead of just reporting 0 or a blank chart

## 1. Install dependencies

In [9]:
!pip install -q streamlit langchain langchain-classic langchain-experimental langchain-groq pandas matplotlib pyngrok

## 2. API keys
- **Groq key** (free, instant, no card) -> https://console.groq.com/keys
- **ngrok authtoken** (free) -> sign up at https://ngrok.com -> dashboard -> "Your Authtoken" (copy it fresh each time, don't reuse an old one)

In [10]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")
os.environ["NGROK_AUTHTOKEN"] = getpass("Enter your ngrok authtoken: ")

Enter your Groq API key: ··········
Enter your ngrok authtoken: ··········


## 3. The agent + app, written to app.py
This is the actual app - csv upload, auto profiling, the sandboxed repl tool the agent uses to run pandas code, the react agent with chat memory, and chart handling, all in one file.

In [11]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os
import glob
import time

from langchain_groq import ChatGroq
from langchain_experimental.tools import PythonAstREPLTool
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate

st.set_page_config(page_title="DataWhisperer", layout="wide")
st.title("📊 DataWhisperer: Talk to Your CSV")
st.caption("Upload a CSV, ask questions in plain English. The agent writes and runs pandas code in a sandbox and returns an answer, table, or chart.")

api_key = st.sidebar.text_input(
    "Groq API Key",
    type="password",
    value=os.environ.get("GROQ_API_KEY", ""),
    help="Get a free key at console.groq.com/keys",
)
uploaded = st.file_uploader("Upload CSV", type=["csv"])

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

if uploaded and api_key:
    df = pd.read_csv(uploaded)
    st.dataframe(df.head(10))

    # need this so the model stops guessing column names/values and gets them wrong
    def build_schema_summary(frame):
        lines = []
        for col in frame.columns:
            dtype = str(frame[col].dtype)
            nulls = int(frame[col].isnull().sum())
            nunique = frame[col].nunique()
            if pd.api.types.is_numeric_dtype(frame[col]):
                col_min = frame[col].min()
                col_max = frame[col].max()
                lines.append(f"- {col} ({dtype}, {nulls} nulls): numeric, range [{col_min}, {col_max}]")
            elif nunique <= 20:
                vals = frame[col].dropna().unique().tolist()
                lines.append(f"- {col} ({dtype}, {nulls} nulls): exact values = {vals}")
            else:
                sample = frame[col].dropna().unique()[:5].tolist()
                lines.append(f"- {col} ({dtype}, {nulls} nulls, {nunique} unique): sample values = {sample}")
        return "\n".join(lines)

    schema_summary = build_schema_summary(df)

    st.subheader("Quick Profile")
    c1, c2, c3 = st.columns(3)
    c1.metric("Rows", df.shape[0])
    c2.metric("Columns", df.shape[1])
    c3.metric("Missing values", int(df.isnull().sum().sum()))

    with st.expander("Column details"):
        st.dataframe(pd.DataFrame({
            "dtype": df.dtypes.astype(str),
            "missing": df.isnull().sum(),
            "unique values": df.nunique(),
        }))
    with st.expander("Summary stats"):
        st.dataframe(df.describe())

    llm = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=api_key, temperature=0)

    # locking the repl down to just these 3 objects so it cant touch disk/files/internet
    safe_locals = {"df": df, "pd": pd, "plt": plt}
    repl_tool = PythonAstREPLTool(locals=safe_locals)
    repl_tool.name = "python_repl"
    repl_tool.description = (
        "A sandboxed Python shell with access ONLY to a pandas dataframe named df, "
        "pandas as pd, and matplotlib.pyplot as plt. Use it to run pandas code that "
        "answers questions about the data. To make a chart, build it with plt and "
        "finish with plt.savefig('chart.png') -- this EXACT filename, nothing else -- "
        "instead of plt.show()."
    )
    tools = [repl_tool]

    prompt = PromptTemplate.from_template("""
You are a careful data analyst agent. A pandas dataframe called df is already loaded (do not reload or redefine it).
Answer the user's question by writing and running pandas/matplotlib code with the python_repl tool.
Always include the exact code you ran in your final answer, for transparency.

Dataset schema (trust this, do not guess or assume column names, dtypes, or spellings --
categorical values are shown EXACTLY as they appear in the data, including capitalization and
spacing, e.g. 'runout' vs 'run out' are NOT the same string, treat them exactly as listed):
{schema}

Rules you must follow:
1. If the question asks to EXCLUDE, IGNORE, or FILTER OUT certain rows, filter the dataframe first
   using boolean indexing BEFORE aggregating. Do NOT use .count() on an unfiltered group as a
   stand-in for a filtered count. Filter first, then aggregate.
2. Before using any column, confirm it exists exactly as spelled in the schema above -- never guess
   or auto-correct a name.
3. Inspect the dtype of every column used in filtering, comparison, sorting, grouping, or
   aggregation before operating on it. If a numeric comparison is needed on a column stored as
   text, convert it safely with pd.to_numeric(..., errors="coerce") rather than comparing strings
   to numbers directly.
4. Numeric columns that encode more than one thing (e.g. "Period" as 2025.12, or a cricket "ball"
   column as 5.3 meaning over 5 ball 3) rarely equal a plain integer or divide cleanly by a fixed
   number. Check a few real values first. For overs specifically: int(ball_value) IS the over, the
   decimal is just the ball count within it -- never invent formulas like (x - 0.1) // 6. Verify any
   such extraction produces a sane number of buckets (~20 for T20, ~50 for ODI) before using it.
5. Only state a number in your Final Answer if you actually saw it printed as an Observation in
   this trace. Never state a number from memory or assumption.
6. After every filter, print the row count (print(len(filtered_df))). If it's 0, empty, or NaN,
   that's a red flag your filter matched nothing -- check the schema for a similar/overlapping
   category name (e.g. "Beer" vs "Total beer") and retry before reporting 0 as a real answer.
7. Print the number of missing values in all columns used for the calculation before aggregating.
   Explicitly decide whether to drop or fill them for this specific question -- never let NaNs
   silently pass through into a sum, mean, or count.
8. Never access .iloc[0], .iat[0], or .values[0] on a dataframe or series without first confirming
   it is non-empty.
9. Never overwrite or reassign the original df -- always assign filtered/transformed results to a
   new variable (e.g. filtered_df = df[...]) so later questions still have the original data intact.
10. You must always give a concrete final answer -- never refuse or say you couldn't verify. If
    running low on steps, use the most recent successfully computed Observation rather than
    starting a new unverified guess.
11. If the user asks for a chart, plot, or trend, actually run the plotting code through the
    python_repl tool as an Action -- don't just describe it. Confirm the dataframe being plotted
    isn't empty first. Call plt.tight_layout() then plt.savefig('chart.png') -- this EXACT
    filename, nothing else, never plt.show(). Only write "Final Answer" after an Observation
    confirms it ran without error.
12. Use the conversation history below to understand follow-up questions (e.g. "now just show me
    the top 3" refers to the previous question's result).
13. For ranking questions ("top", "highest", "lowest", "most", "least"), always sort explicitly
    with sort_values(ascending=True/False). Never rely on the default order returned by groupby(),
    value_counts(), or unique().
14. If multiple rows tie for the requested maximum or minimum value, report all tied rows unless
    the user explicitly asks for only one.
15. Before finalizing, mentally re-check: did I apply every filter/exclusion the question asked
    for, does my final number match an Observation above, and does the row count at each step make
    sense given the question? If unsure, run one more quick sanity check before answering.

Conversation history:
{chat_history}

Tools: {tools}
Tool names: {tool_names}

Use this exact format:
Question: the input question
Thought: your reasoning
Action: one of [{tool_names}]
Action Input: the code to run
Observation: result of the code
... (Thought/Action/Action Input/Observation can repeat)
Thought: I now know the final answer
Final Answer: the answer to the user, including the code you ran

Question: {input}
{agent_scratchpad}
""")

    agent = create_react_agent(llm, tools, prompt)
    executor = AgentExecutor(
        agent=agent, tools=tools, verbose=True,
        handle_parsing_errors=True,
        max_iterations=25,
        max_execution_time=120,
        early_stopping_method="generate",
    )

    for turn in st.session_state.chat_history:
        st.markdown(f"**Q: {turn['q']}**")
        st.write(turn["a"])
        if turn.get("chart"):
            st.image(turn["chart"])

    question = st.text_input("Ask a question about your data", placeholder="e.g. What's the average of column X? Plot a histogram of Y.")
    if st.button("Ask") and question:
        # streamlit reruns the whole script each time but doesnt restart python, so old
        # png files and old matplotlib figures stick around unless we clear them here
        for old_png in glob.glob("*.png"):
            os.remove(old_png)
        plt.close("all")

        history_text = "\n".join(
            f"Q: {t['q']}\nA: {t['a']}" for t in st.session_state.chat_history[-5:]
        ) or "None yet."

        with st.spinner("Thinking..."):
            try:
                result = executor.invoke({"input": question, "chat_history": history_text, "schema": schema_summary})
                answer = result["output"]
                st.markdown("### Answer")
                st.write(answer)

                chart_path = "chart.png" if os.path.exists("chart.png") else None

                if chart_path is None:
                    pngs = glob.glob("*.png")
                    if pngs:
                        chart_path = max(pngs, key=os.path.getctime)

                # spent way too long figuring out charts werent showing - turned out sometimes
                # the agent builds the fig but never calls savefig at all. this grabs it anyway
                if chart_path is None and plt.get_fignums():
                    chart_path = "chart.png"
                    plt.savefig(chart_path, bbox_inches="tight")

                if chart_path:
                    st.image(chart_path)

                with st.expander("Agent trace (debug)"):
                    for i, (action, observation) in enumerate(result.get("intermediate_steps", [])):
                        st.markdown(f"**Step {i+1} — Action:** `{action.tool}`")
                        st.code(action.tool_input, language="python")
                        st.markdown("**Observation:**")
                        st.code(str(observation))

                st.session_state.chat_history.append({"q": question, "a": answer, "chart": chart_path})
            except Exception as e:
                if "rate_limit_exceeded" in str(e) or "429" in str(e):
                    st.error("groq rate limit hit, wait a bit and retry, or use a fresh key")
                else:
                    st.error(f"Error: {e}")
                    with st.expander("Full traceback"):
                        st.exception(e)
else:
    st.info("upload a csv and enter your groq api key to start")

Overwriting app.py


## 4. Run Streamlit + tunnel it out with ngrok
Starts Streamlit in the background, opens an ngrok tunnel, prints the public URL. Click it.

If you get `ERR_NGROK_334` (endpoint already online from a previous run), go to
dashboard.ngrok.com -> Secure Tunnels -> Agents, stop the old session, then re-run this cell.

In [12]:
from pyngrok import ngrok, conf
import time

conf.get_default().auth_token = os.environ["NGROK_AUTHTOKEN"]

# kill any leftover tunnel/streamlit from a previous run, otherwise ngrok throws ERR_NGROK_334
ngrok.kill()
get_ipython().system_raw("pkill -f streamlit")
time.sleep(2)

get_ipython().system_raw("streamlit run app.py --server.port 8501 &> /content/logs.txt &")
time.sleep(4)

public_url = ngrok.connect(8501, "http")
print("Your app is live at:", public_url)

Your app is live at: NgrokTunnel: "https://impeach-matchbox-rejoin.ngrok-free.dev" -> "http://localhost:8501"


## 5. If something breaks
- **Blank page / "site can't be reached"** -> wait 5-10 more seconds and refresh, Streamlit takes a moment to boot.
- **ERR_NGROK_334 (endpoint already online)** -> dashboard.ngrok.com -> Secure Tunnels -> Agents -> stop the old session.
- **ngrok auth error, "authtoken invalid"** -> copy a fresh token from dashboard.ngrok.com/get-started/your-authtoken and re-run cell 2 then cell 4.
- **Groq 429 / rate limited** -> free tier is generous but has a requests-per-minute cap; wait ~30s and retry.
- **Agent errors out on a question** -> run `!cat /content/logs.txt` in a new cell to see the real traceback.
- **Anything looks stuck/broken** -> `Runtime > Restart session`, then re-run cells 1, 2, 3, 4 in order.